# 12 WHCA*：窗口化优先规划


本课按百科条目写。WHCA\* 把 HCA\* 的「一次预约到目标」改成「只保证未来 w 拍」。调度台在最后。

---

## 1. 定义

**WHCA\***（Windowed HCA\*，与 HCA\* 同文，Silver 2005）：仍按全序做时间 A\*，但

- `horizon = w`
- `require_goal = False`（08 课的部分路径）
- 只把长度 ≤ w+1 的前缀写入占用表

适合目标会变、只下发即将执行的一小段。RHCR（14 课）在外层循环里反复调用它。

---

## 2. 和 HCA* / RHCR

| | HCA\* | WHCA\* | RHCR |
|--|--|--|--|
| 窗口 | ∞ | 一次 w 拍 | 每步重规划 w 拍 |
| 到终点 | 必须 | 不必须 | 靠滚动慢慢到 |
| 冲突保证 | 全程 | 仅窗内 | 仅每窗执行的那一步来自无冲突窗 |

不完全、不最优，与 HCA 相同。窗口外路径仍可能交叉——那是下一窗口的事。

---

## 3. 不变量

- `window_paths` pad 到同一长度后，窗内 `is_conflict_free`。
- 单车前缀长度 ≤ w+1（含起点）。
- 远目标：空间最短路 > w 时，必须 `require_goal=False`，否则全员失败。

---

## 4. 伪代码

```
table ← 空
for agent in order:
    π ← TimeA*(..., horizon=w, require_goal=False)
    clip ← π 的前 w+1 个点（失败则原地 pad）
    Reserve(clip)
return window_paths（pad 对齐后再验冲突）
```

---

## 5. 手算

两车各走一行，目标 5 步远，w=3。每人只规划 3 步，停在半路。窗内无交叉。下一窗口从新位置再规划——那是 RHCR。单独跑一次 WHCA **不必**到达。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：裁窗口 + 对齐


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.reservation import ReservationTable
from lib.algos.temporal import time_astar
from lib.algos.prioritized import _reserve_path
from lib.agents import is_conflict_free, pad


def whca_simple(graph, starts, goals, order, window=3):
    table = ReservationTable()
    paths = {}
    for agent in order:
        res = time_astar(
            graph, starts[agent], goals[agent],
            table=table, agent=agent, horizon=window, require_goal=False,
        )
        print(agent, "found", res.found, "path", res.path, "partial", res.extra)
        if not res.found:
            paths[agent] = [starts[agent]] * (window + 1)
            continue
        clip = res.path[: window + 1]
        paths[agent] = clip
        _reserve_path(table, clip, agent)
    window_paths = {a: pad(p, window + 1)[: window + 1] for a, p in paths.items()}
    return {
        "found": is_conflict_free(window_paths),
        "paths": paths,
        "window_paths": window_paths,
        "window": window,
    }


g, starts, goals = two_agent_tunnel()
r = whca_simple(g, starts, goals, ["a", "b"], window=3)
print("窗内无冲突", r["found"])
print("窗路径", r["window_paths"])


## 7. 逐行

| 参数 | 作用 |
|------|------|
| `horizon=window` | 状态 t 不超过 w |
| `require_goal=False` | 允许部分路径 |
| `clip[:window+1]` | 含起点共 w+1 个点 = w 步 |
| `pad` | 对齐长度再查窗内冲突 |

对比：同样窗口若 `require_goal=True`，到不了的车会失败。


In [ ]:
from lib.algos.prioritized import whca
from lib.algos.temporal import time_astar
from lib.reservation import ReservationTable

lib_r = whca(g, starts, goals, window=3)
print("库 WHCA found", lib_r["found"], lib_r["window_paths"])

# 反例：必须到终点 + 短窗口
bad = time_astar(g, starts["a"], goals["a"], horizon=3, require_goal=True)
print("a 要走 5 步、horizon=3、require_goal 时 found", bad.found)


## 常见 bug

1. horizon 小于最短步数且 require_goal=True ⇒ 全失败。
2. 窗口路径不 pad，冲突检测对不齐时间。
3. 把窗内前缀当成全程解，评测「到没到终点」会误判。
4. 窗口外不管，却用全程 `is_conflict_free` 当成功标准。

## 条目小结

| 项目 | 内容 |
|------|------|
| 窗口 | 只保证未来 w 拍 |
| 到终点 | 不要求 |
| 下一课 | 优先级不要预先固定 → PBS |

## 练习

1. w=1 时每车只走一步。两车对头在无向走廊，w=1 够不够换位？
2. 解释 `window+1`：w 步为什么有 w+1 个点。
3. 为何 RHCR 每步重规划，而不是一次 WHCA 把窗内全走完？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), lib_r, "WHCA*")
studio
